In [2]:
import numpy as np

# ---------------------------------------------------------------- standard attention
def softmax_stable(S):
    m = S.max(axis=1, keepdims=True)           # subtract row max: exp args <= 0, no overflow
    m = np.where(np.isfinite(m), m, 0.0)       # guard fully-masked rows
    e = np.exp(S - m)
    return e / e.sum(axis=1, keepdims=True)

def softmax_naive(S):
    e = np.exp(S)
    return e / e.sum(axis=1, keepdims=True)

def attention_forward(Q, K, V, causal=False, stable=True):
    n, d = Q.shape
    S = Q @ K.T / np.sqrt(d)
    if causal:
        S = np.where(np.tril(np.ones((n, n), dtype=bool)), S, -np.inf)
    P = (softmax_stable if stable else softmax_naive)(S)
    O = P @ V
    return O, (Q, K, V, P)

def attention_backward(dO, cache):
    Q, K, V, P = cache
    d = Q.shape[1]
    dV = P.T @ dO
    dP = dO @ V.T
    dS = P * (dP - np.sum(dP * P, axis=1, keepdims=True))   # masked entries have P=0 -> dS=0
    dQ = dS @ K / np.sqrt(d)
    dK = dS.T @ Q / np.sqrt(d)
    return dQ, dK, dV

# ---------------------------------------------------------------- gradient check
def numerical_grad(f, x, eps):
    g = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

def rel_err(a, b):
    return np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-30)

def gradcheck(n=6, d=4, causal=False, eps=1e-6, tol=1e-6, seed=0):
    rng = np.random.default_rng(seed)
    Q, K, V = (rng.standard_normal((n, d)) for _ in range(3))      # float64
    W = rng.standard_normal((n, d))                                 # loss L = sum(O * W)  => dO = W
    def loss():
        O, _ = attention_forward(Q, K, V, causal=causal)
        return float(np.sum(O * W))
    O, cache = attention_forward(Q, K, V, causal=causal)
    dQ, dK, dV = attention_backward(W, cache)
    res = {}
    for name, x, g in [("dQ", Q, dQ), ("dK", K, dK), ("dV", V, dV)]:
        res[name] = rel_err(numerical_grad(loss, x, eps), g)
    ok = all(v < tol for v in res.values())
    return res, ok

# ---------------------------------------------------------------- FlashAttention-style tiling
def flash_forward(Q, K, V, causal=False, Bq=16, Bk=16):
    """Online softmax. Memory: O(n*d) + O(Bq*Bk) tiles + O(n) logsumexp. Never forms n x n."""
    n, d = Q.shape; scale = 1 / np.sqrt(d)
    O = np.zeros_like(Q); L = np.zeros(n)
    for i0 in range(0, n, Bq):
        i1 = min(i0 + Bq, n); Qi = Q[i0:i1]
        m = np.full(i1 - i0, -np.inf); l = np.zeros(i1 - i0); acc = np.zeros((i1 - i0, d))
        for j0 in range(0, n, Bk):
            j1 = min(j0 + Bk, n)
            if causal and j0 > i1 - 1:       # whole tile is above the diagonal -> skip (saves ~half FLOPs)
                continue
            S = Qi @ K[j0:j1].T * scale
            if causal:
                rows = np.arange(i0, i1)[:, None]; cols = np.arange(j0, j1)[None, :]
                S = np.where(cols <= rows, S, -np.inf)
            m_new = np.maximum(m, S.max(axis=1))
            P = np.exp(S - m_new[:, None])
            alpha = np.exp(m - m_new)                       # rescale old running sums
            l = alpha * l + P.sum(axis=1)
            acc = alpha[:, None] * acc + P @ V[j0:j1]
            m = m_new
        O[i0:i1] = acc / l[:, None]; L[i0:i1] = m + np.log(l)
    return O, L

def flash_backward(Q, K, V, O, L, dO, causal=False, Bq=16, Bk=16):
    """Recomputes P tile-by-tile from saved logsumexp L (O(n)) instead of storing P (O(n^2))."""
    n, d = Q.shape; scale = 1 / np.sqrt(d)
    dQ = np.zeros_like(Q); dK = np.zeros_like(K); dV = np.zeros_like(V)
    D = np.sum(dO * O, axis=1)                              # D_i = rowsum(dO*O) = rowsum(dP*P)
    for j0 in range(0, n, Bk):
        j1 = min(j0 + Bk, n); Kj, Vj = K[j0:j1], V[j0:j1]
        for i0 in range(0, n, Bq):
            i1 = min(i0 + Bq, n)
            if causal and j0 > i1 - 1:
                continue
            S = Q[i0:i1] @ Kj.T * scale
            if causal:
                rows = np.arange(i0, i1)[:, None]; cols = np.arange(j0, j1)[None, :]
                S = np.where(cols <= rows, S, -np.inf)
            P = np.exp(S - L[i0:i1, None])                  # exact softmax probabilities, recomputed
            dV[j0:j1] += P.T @ dO[i0:i1]
            dP = dO[i0:i1] @ Vj.T
            dS = P * (dP - D[i0:i1, None])
            dQ[i0:i1] += dS @ Kj * scale
            dK[j0:j1] += dS.T @ Q[i0:i1] * scale
    return dQ, dK, dV

# ---------------------------------------------------------------- demos
def stability_demo():
    rng = np.random.default_rng(0)
    S = rng.standard_normal((3, 5)) * 5 + 800.0               # large logits -> exp(800) = inf in float64
    with np.errstate(over="ignore", invalid="ignore"):
        naive = softmax_naive(S)
    print("naive softmax of logits ~800   ->", np.round(naive[0], 3), "(inf/inf = NaN)")
    print("stable softmax (subtract max)  ->", np.round(softmax_stable(S)[0], 3), " row sum =", softmax_stable(S)[0].sum())
    S32 = (rng.standard_normal((1, 5)) * 5 + 100).astype(np.float32)
    with np.errstate(over="ignore", invalid="ignore"):
        print("float32 naive at logits ~100   ->", softmax_naive(S32)[0], "(overflows already at ~88.7)")

def scaling_demo(d_list=(16, 64, 256, 1024), n=64, trials=20):
    rng = np.random.default_rng(1)
    print(f"{'d':>6} | {'Var(q.k) unscaled':>18} | {'Var scaled':>11} | {'max P unscaled':>14} | {'max P scaled':>12} | {'mean |dS| unscaled':>18} | {'scaled':>8}")
    for d in d_list:
        v_u, mp_u, mp_s, g_u, g_s = [], [], [], [], []
        for _ in range(trials):
            Q, K = rng.standard_normal((n, d)), rng.standard_normal((n, d))
            Su = Q @ K.T; Ss = Su / np.sqrt(d)
            Pu, Ps = softmax_stable(Su), softmax_stable(Ss)
            v_u.append(Su.var()); mp_u.append(Pu.max(1).mean()); mp_s.append(Ps.max(1).mean())
            dP = rng.standard_normal((n, n))                  # generic upstream signal
            g_u.append(np.abs(Pu * (dP - (dP * Pu).sum(1, keepdims=True))).mean())
            g_s.append(np.abs(Ps * (dP - (dP * Ps).sum(1, keepdims=True))).mean())
        print(f"{d:>6} | {np.mean(v_u):>18.1f} | {np.mean(v_u)/d:>11.2f} | {np.mean(mp_u):>14.3f} | {np.mean(mp_s):>12.3f} | {np.mean(g_u):>18.5f} | {np.mean(g_s):>8.5f}")

def memory_table():
    print(f"{'n':>8} | {'P (fp16) per head':>18} | {'P+S+dP+dS fp32 /head':>22} | {'Flash extras (L,m,l) fp32':>26}")
    for n in [1024, 4096, 16384, 65536, 131072]:
        P16 = n * n * 2; four32 = 4 * n * n * 4; fl = 3 * n * 4
        f = lambda b: f"{b/2**20:,.1f} MiB" if b < 2**30 else f"{b/2**30:,.1f} GiB"
        print(f"{n:>8} | {f(P16):>18} | {f(four32):>22} | {f(fl):>26}")

if __name__ == "__main__":
    print("== Gradient check (float64, central difference) ==")
    EPS, TOL = 1e-6, 1e-6
    print(f"step size h = {EPS:g}, precision = float64 (eps_mach 2.2e-16), accept if relative error < {TOL:g}")
    print("relative error = ||g_num - g_ana|| / (||g_num|| + ||g_ana||)")
    for causal in (False, True):
        res, ok = gradcheck(causal=causal, eps=EPS, tol=TOL)
        print(f"  causal={causal!s:<5}", {k: f"{v:.2e}" for k, v in res.items()}, "PASS" if ok else "FAIL")
        assert ok
    # multiple seeds / shapes
    worst = max(max(gradcheck(n=n, d=d, causal=c, seed=s)[0].values()) for n in (3, 8) for d in (2, 5) for c in (False, True) for s in range(5))
    print(f"  worst rel. error over 40 random configs: {worst:.2e}")

    print("\n== Causal mask check: masked entries get exactly zero gradient & no future leakage ==")
    rng = np.random.default_rng(3); n, d = 7, 4
    Q, K, V = (rng.standard_normal((n, d)) for _ in range(3)); dO = np.zeros((n, d)); dO[2] = 1.0   # loss depends only on output row 2
    O, c = attention_forward(Q, K, V, causal=True); dQ, dK, dV = attention_backward(dO, c)
    print("  loss on output row 2 -> dK/dV rows 3..6 are exactly zero:", bool(np.all(dK[3:] == 0) and np.all(dV[3:] == 0)), "| dQ rows !=2 zero:", bool(np.all(np.delete(dQ, 2, 0) == 0)))

    print("\n== Stability issue: naive softmax overflow ==")
    stability_demo()
    print("\n== Why 1/sqrt(d) ==")
    scaling_demo()

    print("\n== Tiled (FlashAttention-style) vs. standard: forward and backward ==")
    rng = np.random.default_rng(5)
    for causal in (False, True):
        n, d = 70, 8   # n not a multiple of the block size on purpose
        Q, K, V, dO = (rng.standard_normal((n, d)) for _ in range(4))
        O, cache = attention_forward(Q, K, V, causal=causal); g_ref = attention_backward(dO, cache)
        Of, L = flash_forward(Q, K, V, causal=causal, Bq=16, Bk=16)
        g_fl = flash_backward(Q, K, V, Of, L, dO, causal=causal, Bq=16, Bk=16)
        errs = [np.abs(Of - O).max()] + [np.abs(a - b).max() for a, b in zip(g_fl, g_ref)]
        print(f"  causal={causal!s:<5} max|diff| O, dQ, dK, dV =", [f"{e:.1e}" for e in errs], "PASS" if max(errs) < 1e-10 else "FAIL")
        assert max(errs) < 1e-10

    print("\n== Memory vs sequence length (per head, per batch element) ==")
    memory_table()


== Gradient check (float64, central difference) ==
step size h = 1e-06, precision = float64 (eps_mach 2.2e-16), accept if relative error < 1e-06
relative error = ||g_num - g_ana|| / (||g_num|| + ||g_ana||)
  causal=False {'dQ': '1.90e-10', 'dK': '4.07e-10', 'dV': '9.05e-11'} PASS
  causal=True  {'dQ': '2.37e-10', 'dK': '3.19e-10', 'dV': '1.06e-10'} PASS
  worst rel. error over 40 random configs: 1.85e-09

== Causal mask check: masked entries get exactly zero gradient & no future leakage ==
  loss on output row 2 -> dK/dV rows 3..6 are exactly zero: True | dQ rows !=2 zero: True

== Stability issue: naive softmax overflow ==
naive softmax of logits ~800   -> [nan nan nan nan nan] (inf/inf = NaN)
stable softmax (subtract max)  -> [0.065 0.018 0.856 0.059 0.002]  row sum = 1.0000000000000002
float32 naive at logits ~100   -> [nan nan nan nan nan] (overflows already at ~88.7)

== Why 1/sqrt(d) ==
     d |  Var(q.k) unscaled |  Var scaled | max P unscaled | max P scaled | mean |dS| unscaled